# Exact Reconstruction of the EV Adoption Source Dataset

## What this notebook proves

The 10,000-row dataset that inspired Playground Series S6E9 can be recreated with
NumPy's legacy `RandomState(101)`. The code below reproduces every ID, feature,
target, and missing value: **150,000 cells in total, with zero mismatches**.

This is an exact reconstruction of the public
[EV Adoption Behavior and Range Anxiety dataset](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety),
not of the competition train and test files. The competition data clearly went
through another process, so this notebook does not claim to identify Kaggle's generator.

## How I found it

I worked from simple clues to strict tests:

1. I measured each column's range, frequencies, rounding, and missing values.
   Income stops at 30,000 and commute distance stops at 5.0, both with the shape
   of a normal distribution whose left tail was clipped.
2. Simple round parameters matched: income uses `Normal(85000, 35000)` and commute
   uses `Normal(40, 25)`. The category frequencies also reduce to round probabilities.
3. Charging infrastructure has different integer ranges for Urban, Suburban, and
   Rural rows. Exact matching requires drawing three values row by row in this order:
   home stations, work stations, then home charging.
4. Range anxiety and the purchase target each come from a short weighted score plus
   one standard-normal noise draw. I added plausible inputs one at a time and solved
   the remaining mismatches until the thresholds and coefficients were exact.
5. The three 2% missing-value masks are drawn only after every feature and target
   has been created.
6. Finally, I froze the complete recipe and compared all 10,000 × 15 cells. This is
   much stronger than merely matching summary statistics: one extra random draw
   would shift the random stream and break almost everything after it.

### A closer look: how the `buy_score` formula was identified

The coefficients in `buy_score` were not guessed all at once. There were two
separate stages: first estimating the shape of the rule, then recovering its exact
implementation.

**Why suspect a hidden score in the first place?** The target has only two visible
values, Yes and No, but its rates change smoothly and monotonically with quantities
such as income and environmental concern. Subsidy and anxiety shift those rates in
consistent directions. This is the pattern we would expect if each row first
received a continuous "willingness to buy" score and the answer became Yes only
when that score crossed a threshold. It is also a common, simple way to write a
synthetic-data script: add a few understandable effects, add personal randomness,
and apply one cutoff.

**1. Find the small set of relevant columns.** I first made simple grouped checks,
such as purchase rate by concern level, subsidy status, anxiety category, and income
band. This was not yet model fitting. It answered two more basic questions: which
variables move the rate clearly, and whether their direction makes real-world
sense. Income and concern increased the rate, subsidy caused an upward shift, and
lower anxiety caused a large upward shift. Once these four variables were included,
the other columns were not needed to reproduce the source dataset's short purchase
rule. This suggested an additive score rather than a large tree of special cases.

**Why try probit rather than jump straight to exact numbers?** At this point the
unknown script could still have used a logistic probability, a Gaussian-noise
threshold, or some other monotone rule. Logistic and probit models are useful
diagnostic tools because both turn a binary label back into an estimated linear
score. Probit has a particularly direct interpretation:

$$Y=1[S+z>0],\qquad z\sim N(0,1),\qquad P(Y=1\mid X)=\Phi(S).$$

The already reconstructed feature code repeatedly used NumPy normal draws, so one
more normal draw for individual purchase randomness was a natural small-script
hypothesis. On the source data, probit and logistic ranking performance was almost
identical, but probit had slightly better probability loss. That was supporting
evidence, not proof; the exact replay below supplied the proof.

**2. Fit an approximate probit model.** Using High anxiety as the reference level,
I fitted the additive terms suggested by the grouped checks. The coefficients were
approximately:

| Term | Probit estimate | Recovered value |
|---|---:|---:|
| Intercept | -8.0173 | -8.5 |
| Income / 10,000 | 0.13054 | 0.12 |
| Environmental concern | 0.61475 | 0.6 |
| Subsidy = Yes | 1.89653 | 2.0 |
| Range anxiety = Low | 2.45986 | 3.0 |
| Range anxiety = Medium | 1.48255 | 2.0 |

These estimates did not prove the exact formula, but they placed every coefficient
close to a simple, human-chosen number. Dividing income by 10,000 was also natural:
it gives a human-sized coefficient instead of a tiny effect per dollar. The fitted
values were treated as a map showing where to search, not as the final generator.

**How did an approximate statistical model become an exact reconstruction?** The
feature investigation had already found something unusually strong: starting from
`RandomState(101)` and replaying Age, Gender, Income, City, and the remaining
features in order reproduced their actual row values. A pseudo-random generator is
deterministic. If the seed and every earlier call are the same, its internal state
is the same too. Therefore the next numbers in its stream can be recovered exactly.

**3. Recover the exact noise values.** After replaying all earlier feature calls,
the next `rng.normal(size=10_000)` produces the 10,000 values that the source script
could have used for purchase randomness. These are not fitted residuals and are not
fresh random substitutes: they are the exact next values in the recovered stream.
If even one earlier random call were missing or out of order, this vector would be
different and the label reconstruction would fail widely.

**4. Turn every label into an inequality.** For row $i$, let $S_i$ be the part of
the score without noise and let $z_i$ be the recovered noise. A Yes label requires
$S_i + z_i > 0$; a No label requires $S_i + z_i \leq 0$. The dataset therefore
supplies 10,000 simultaneous constraints on the coefficients.

**Why do the inequalities reveal more than regression?** Regression only asks which
coefficients make the observed labels probable on average. Once each $z_i$ is
known, every label gives a hard boundary. For example, a Yes row says that its
deterministic score must be greater than $-z_i$, while a No row says it must not be.
A candidate that looks statistically excellent can still fail one of these exact
boundaries.

**5. Test simple coefficients near the fitted values.** Round, script-like values
near the probit estimates were checked against all 10,000 hard inequalities. This
was a small, well-directed search because the earlier fit had already located each
coefficient. The values shown in the right-hand column satisfy every row when
combined with the recovered noise stream.

This is why the result is stronger than a regression model with a good score. The
recovered formula reproduces all 10,000 binary labels exactly; continuing the same
random stream also reproduces the later missing-value masks. Finally, the complete
replay matches all 150,000 cells. Algebraically equivalent ways to write the same
score can still exist, and this result applies only to the 10,000-row source dataset,
not to the separate competition-data generator.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 101
COLUMNS = [
    "Buyer_ID",
    "Age",
    "Gender",
    "Annual_Income_USD",
    "City_Type",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Current_Car_Type",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Home_Charging_Possible",
    "Environmental_Concern_Level",
    "Subsidy_Available",
    "Range_Anxiety_Level",
    "Will_Buy_EV",
]


def make_city_infrastructure(rng, city):
    """Draw three city-dependent values, row by row, in the original order."""
    home = np.empty(len(city), dtype=int)
    work = np.empty(len(city), dtype=int)
    can_charge = np.empty(len(city), dtype="<U3")
    rules = {
        "Urban": (2, 15, 3, 20, 0.4),
        "Suburban": (0, 8, 1, 10, 0.8),
        "Rural": (0, 3, 0, 4, 0.9),
    }

    for i, city_name in enumerate(city):
        home_low, home_high, work_low, work_high, yes_rate = rules[city_name]
        home[i] = rng.randint(home_low, home_high)
        work[i] = rng.randint(work_low, work_high)
        can_charge[i] = "Yes" if rng.rand() < yes_rate else "No"
    return home, work, can_charge


def generate_source_data(rows=10_000, seed=SEED):
    """Recreate the source dataset using the original random-draw order."""
    rng = np.random.RandomState(seed)

    age = rng.randint(25, 70, rows)
    gender = rng.choice(["Male", "Female", "Other"], rows, p=[0.52, 0.45, 0.03])
    income = np.clip(rng.normal(85_000, 35_000, rows), 30_000, None).astype(int)
    city = rng.choice(["Urban", "Suburban", "Rural"], rows, p=[0.50, 0.35, 0.15])
    cars = rng.choice([1, 2, 3, 4], rows, p=[0.40, 0.40, 0.15, 0.05])
    car_type = rng.choice(
        ["Sedan", "SUV", "Hatchback", "Truck"],
        rows,
        p=[0.40, 0.35, 0.15, 0.10],
    )
    commute = np.round(np.clip(rng.normal(40, 25, rows), 5, None), 1)

    stations_home, stations_work, can_charge = make_city_infrastructure(rng, city)
    concern = rng.randint(1, 6, rows)
    subsidy = rng.choice(["Yes", "No"], rows, p=[0.60, 0.40])

    anxiety_score = (
        0.02 * commute
        - 0.1 * (stations_home + stations_work)
        - 3.0 * (can_charge == "Yes")
        + rng.normal(size=rows)
    )
    anxiety = np.where(
        anxiety_score < -0.5,
        "Low",
        np.where(anxiety_score < 1.5, "Medium", "High"),
    )

    buy_score = (
        -8.5
        + 0.12 * (income / 10_000)
        + 0.6 * concern
        + 2.0 * (subsidy == "Yes")
        + 3.0 * (anxiety == "Low")
        + 2.0 * (anxiety == "Medium")
        + rng.normal(size=rows)
    )
    will_buy = np.where(buy_score > 0, "Yes", "No")

    # Missingness is injected last, independently, into these three columns.
    income = income.astype(float)
    commute = commute.copy()
    concern = concern.astype(float)
    income[rng.rand(rows) < 0.02] = np.nan
    commute[rng.rand(rows) < 0.02] = np.nan
    concern[rng.rand(rows) < 0.02] = np.nan

    return pd.DataFrame(
        {
            "Buyer_ID": [f"EV{i:05d}" for i in range(1, rows + 1)],
            "Age": age,
            "Gender": gender,
            "Annual_Income_USD": income,
            "City_Type": city,
            "Daily_Commute_km": commute,
            "Number_of_Cars_Owned": cars,
            "Current_Car_Type": car_type,
            "Charging_Stations_Near_Home": stations_home,
            "Charging_Stations_Near_Work": stations_work,
            "Home_Charging_Possible": can_charge,
            "Environmental_Concern_Level": concern,
            "Subsidy_Available": subsidy,
            "Range_Anxiety_Level": anxiety,
            "Will_Buy_EV": will_buy,
        },
        columns=COLUMNS,
    )


def count_mismatches(original, reproduced):
    """Count cell mismatches by column, treating two missing values as equal."""
    assert original.columns.tolist() == reproduced.columns.tolist()
    assert len(original) == len(reproduced)

    result = {}
    for column in COLUMNS:
        left = original[column].to_numpy()
        right = reproduced[column].to_numpy()
        same = (left == right) | (pd.isna(left) & pd.isna(right))
        result[column] = int((~same).sum())
    return result

## Verify every cell

The original file is attached from the public CC0 dataset. No competition data is used.

In [ ]:
filename = "EV_Adoption_and_Range_Anxiety_Dataset.csv"
kaggle_input = Path("/kaggle/input")
matches = list(kaggle_input.rglob(filename)) if kaggle_input.is_dir() else []
if not kaggle_input.is_dir():  # Local fallback for reproducibility outside Kaggle.
    matches = [Path("data/external/EV_Adoption_and_Range_Anxiety_Dataset.csv")]
assert len(matches) == 1, f"Expected one attached source file, found: {matches}"

original = pd.read_csv(matches[0])
reproduced = generate_source_data(rows=len(original))
mismatches = count_mismatches(original, reproduced)

print(f"Rows: {len(original):,}")
print(f"Columns: {len(original.columns)}")
print(f"Cells compared: {original.size:,}")
print(f"Mismatched cells: {sum(mismatches.values()):,}")
assert sum(mismatches.values()) == 0

## Result and boundary

Expected output: 10,000 rows, 15 columns, 150,000 cells compared, and 0 mismatches.
The assertion makes the notebook fail loudly if any value differs.

The source file is CC0 1.0. Its SHA-256 is
`c271a380df51b18177d0a039d54525ca7c3d71500701dc7496714a091df16fae`.
The reconstruction was verified with Python 3.12, NumPy 2.5.2, and pandas 3.0.5.

One final caution: about 97.94% of competition income values lie on the source
dataset's finite value support, which is incompatible with simply rerunning this
continuous formula using a different seed. The exact competition generator remains unknown.